# 03c · End to end CNN reference

**Purpose.** Train the same CNN end to end as a classifier on both datasets and both tasks, same folds and seeds. Reported separately as a reference point, not as a fourth representation.

**Inputs**
- GasWindTunnel (600, 72, 1500) and EverydayScents (144, 11, 210) + both `meta.csv`

**Outputs**
- `results/03c_end_to_end_reference.csv`: one value per seed, fold, dataset, task

## Colab setup
Run this first on Google Colab (Runtime → Change runtime type → **GPU**). It clones the repo, downloads the
processed data from the shared Drive folder into `data/processed/`, and moves into this notebook's folder so
that the rest of the notebook runs unchanged. On a laptop it does nothing.

In [1]:
import glob
import os
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
REPO = "/content/gas-repr-guide"
REPO_URL = "https://github.com/YasserAlbogami/gas-repr-guide.git"
DATA_URL = "https://drive.google.com/drive/folders/1rNIcZkHDFhH2dMVq8TgNWt_0IH2yw2_I"   # processed data (Yasser)
DATA_DIRS = ["gas_wind_tunnel", "everyday_scents"]

if IN_COLAB:
    # 1. code (src/). Tried in this order:
    #    a) clone the GitHub repo (works when the repo is public)
    #    b) clone with a GitHub token saved in Colab Secrets (key icon on the left) as GITHUB_TOKEN
    #    c) src.zip uploaded to /content (Files panel on the left)
    if not os.path.exists(f"{REPO}/src"):
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")           # fail instead of asking for a password
        ok = subprocess.run(["git", "clone", "-q", REPO_URL, REPO], env=env).returncode == 0
        if not ok:
            token = None
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                pass
            if token:
                url = REPO_URL.replace("https://", f"https://{token}@")
                ok = subprocess.run(["git", "clone", "-q", url, REPO], env=env).returncode == 0
        if not ok and os.path.exists("/content/src.zip"):
            os.makedirs(f"{REPO}/notebooks/03_encoder", exist_ok=True)
            shutil.unpack_archive("/content/src.zip", REPO)
            ok = True
        if not ok:
            raise RuntimeError("Could not get the code: the repo is private. Make it public, or add GITHUB_TOKEN "
                               "in Colab Secrets, or upload src.zip to /content, then run this cell again.")
    elif os.path.exists(f"{REPO}/.git"):
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=False)
    os.makedirs(f"{REPO}/results", exist_ok=True)

    # 2. processed data (only downloaded once per Colab session)
    processed = f"{REPO}/data/processed"
    if not all(os.path.exists(f"{processed}/{d}/X_signals.npy") for d in DATA_DIRS):
        import gdown
        tmp = "/content/processed_download"
        gdown.download_folder(DATA_URL, output=tmp, quiet=False)
        for d in DATA_DIRS:
            found = glob.glob(f"{tmp}/**/{d}", recursive=True)
            assert found, f"folder {d} not found in the Drive download"
            shutil.copytree(found[0], f"{processed}/{d}", dirs_exist_ok=True)
    for d in DATA_DIRS:
        files = sorted(os.listdir(f"{processed}/{d}"))
        print(d, files)
        for f in ["X_signals.npy", "meta.csv", "info.json"]:
            assert f in files, f"{d}/{f} is missing"

    # 3. run the rest of the notebook from its own folder, as on a laptop (so ../.. = project root)
    os.chdir(f"{REPO}/notebooks/03_encoder")

    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
          else "NONE, switch on Runtime > Change runtime type > GPU")

Retrieving folder contents


Retrieving folder 1YSf-59uCQxHrruY705o36CXtRQ0XS2Fr everyday_scents
Processing file 1hdYRGs6ijl-mYorzn2esfZaWM_Tnd7Ey baseline_level.npy
Processing file 1oPfagu_BzcC2ZvL2SX7bNG8_UDIGygZv info.json
Processing file 1skq8t0lHKIjaht2QFEVxxVZCJf7Wqfvf info.json
Processing file 1KHEy1xcDVFGO3ipc7kExOHk7rUtNCKkr meta.csv
Processing file 1LGP9oFexIJepxC2ZEzbDUAgw2xAYoBeL X_signals.npy
Retrieving folder 1WOzILzMT3EEu4nsar-gg1IH0GNSse4x5 gas_wind_tunnel
Processing file 1Wh2umVgpw82YCugVFMEBM8g6ropQYpvr baseline_level.npy
Processing file 1TRpQKW_t8W5liucNyAt84_I7K2At7QB2 info.json
Processing file 1eX05vgphoRl020zxcypuktJ2OEKstp6O meta.csv
Processing file 128GyQQJ_BsxwQI3fZDSBSEeHhQvtDlGE X_signals.npy


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1hdYRGs6ijl-mYorzn2esfZaWM_Tnd7Ey
To: /content/processed_download/everyday_scents/baseline_level.npy
100%|██████████| 6.46k/6.46k [00:00<00:00, 21.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1oPfagu_BzcC2ZvL2SX7bNG8_UDIGygZv
To: /content/processed_download/everyday_scents/info.json
100%|██████████| 297/297 [00:00<00:00, 1.08MB/s]
Downloading...
From: https://drive.google.com/uc?id=1skq8t0lHKIjaht2QFEVxxVZCJf7Wqfvf
To: /content/processed_download/everyday_scents/info.json
100%|██████████| 297/297 [00:00<00:00, 1.12MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KHEy1xcDVFGO3ipc7kExOHk7rUtNCKkr
To: /content/processed_download/everyday_scents/meta.csv
100%|██████████| 26.7k/26.7k [00:00<00:00, 38.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1LGP9oFexIJepxC2ZEzbDUAgw2xAYoBeL
To: /content/processe

gas_wind_tunnel ['.gitkeep', 'X_signals.npy', 'baseline_level.npy', 'info.json', 'meta.csv']
everyday_scents ['.gitkeep', 'X_signals.npy', 'baseline_level.npy', 'info.json', 'meta.csv']
GPU: Tesla T4


In [2]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import TASKS, load_processed, save_results, summarize
from src.features.encoder import run_end_to_end

NAME = "03c_end_to_end_reference"

## The same CNN used directly as the classifier
Same folds and seeds as the other notebooks; trained on the training fold only. Reported as a reference point,
not as a fourth representation (it has no separate head).

In [3]:
scores, preds = [], []
for dataset in ["gwt", "es"]:
    X, meta, info = load_processed(dataset)          # (N, C, T)
    for task in TASKS:
        s, p = run_end_to_end(X, meta, dataset, task)
        scores.append(s)
        preds.append(p)
scores, preds = pd.concat(scores, ignore_index=True), pd.concat(preds, ignore_index=True)
save_results(scores, preds, NAME)

gwt multi end_to_end seed 0 fold 0: 0.967
gwt multi end_to_end seed 0 fold 1: 0.984
gwt multi end_to_end seed 0 fold 2: 1.000
gwt multi end_to_end seed 0 fold 3: 0.949
gwt multi end_to_end seed 0 fold 4: 0.957
gwt multi end_to_end seed 1 fold 0: 0.983
gwt multi end_to_end seed 1 fold 1: 0.959
gwt multi end_to_end seed 1 fold 2: 0.983
gwt multi end_to_end seed 1 fold 3: 0.951
gwt multi end_to_end seed 1 fold 4: 0.967
gwt multi end_to_end seed 2 fold 0: 0.992
gwt multi end_to_end seed 2 fold 1: 0.983
gwt multi end_to_end seed 2 fold 2: 0.957
gwt multi end_to_end seed 2 fold 3: 0.975
gwt multi end_to_end seed 2 fold 4: 0.967
gwt binary end_to_end seed 0 fold 0: 1.000
gwt binary end_to_end seed 0 fold 1: 1.000
gwt binary end_to_end seed 0 fold 2: 1.000
gwt binary end_to_end seed 0 fold 3: 0.922
gwt binary end_to_end seed 0 fold 4: 0.978
gwt binary end_to_end seed 1 fold 0: 1.000
gwt binary end_to_end seed 1 fold 1: 0.954
gwt binary end_to_end seed 1 fold 2: 1.000
gwt binary end_to_end seed

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


es multi end_to_end seed 0 fold 1: 0.559
es multi end_to_end seed 0 fold 2: 0.756
es multi end_to_end seed 0 fold 3: 0.468
es multi end_to_end seed 0 fold 4: 0.827
es multi end_to_end seed 1 fold 0: 0.083
es multi end_to_end seed 1 fold 1: 0.578
es multi end_to_end seed 1 fold 2: 0.883
es multi end_to_end seed 1 fold 3: 0.927
es multi end_to_end seed 1 fold 4: 1.000
es multi end_to_end seed 2 fold 0: 0.967
es multi end_to_end seed 2 fold 1: 0.232
es multi end_to_end seed 2 fold 2: 0.565
es multi end_to_end seed 2 fold 3: 0.590
es multi end_to_end seed 2 fold 4: 0.649
es binary end_to_end seed 0 fold 0: 1.000
es binary end_to_end seed 0 fold 1: 0.915
es binary end_to_end seed 0 fold 2: 0.931
es binary end_to_end seed 0 fold 3: 0.933
es binary end_to_end seed 0 fold 4: 0.924
es binary end_to_end seed 1 fold 0: 1.000
es binary end_to_end seed 1 fold 1: 0.917
es binary end_to_end seed 1 fold 2: 1.000
es binary end_to_end seed 1 fold 3: 0.880
es binary end_to_end seed 1 fold 4: 1.000
es bin

In [4]:
for metric in ["macro_f1", "balanced_acc", "roc_auc"]:
    print(metric)
    display(summarize(scores.dropna(subset=[metric]), metric))

macro_f1


head                             end_to_end
dataset task   rep                         
es      binary R3_end_to_end  0.957 ± 0.041
        multi  R3_end_to_end  0.662 ± 0.265
gwt     binary R3_end_to_end  0.964 ± 0.058
        multi  R3_end_to_end  0.972 ± 0.016

balanced_acc


head                             end_to_end
dataset task   rep                         
es      binary R3_end_to_end  0.955 ± 0.047
        multi  R3_end_to_end  0.710 ± 0.222
gwt     binary R3_end_to_end  0.972 ± 0.040
        multi  R3_end_to_end  0.972 ± 0.015

roc_auc


,,head,end_to_end
dataset,task,rep,
es,binary,R3_end_to_end,0.984 ± 0.030
gwt,binary,R3_end_to_end,0.985 ± 0.027


## Colab: save the results
Colab deletes everything when the session ends. This copies the results csv files to your Google Drive
(`MyDrive/gas-repr-guide/results/`) and also downloads them to your computer. On a laptop it does nothing.

In [5]:
if IN_COLAB:
    out_files = sorted(glob.glob(f"{REPO}/results/{NAME}*.csv"))
    print(out_files)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        out_dir = "/content/drive/MyDrive/gas-repr-guide/results"
        os.makedirs(out_dir, exist_ok=True)
        for f in out_files:
            shutil.copy(f, out_dir)
        print("copied to", out_dir)
    except Exception as e:
        print("Drive copy failed:", e)
    from google.colab import files
    for f in out_files:
        files.download(f)

['/content/gas-repr-guide/results/03c_end_to_end_reference.csv', '/content/gas-repr-guide/results/03c_end_to_end_reference_preds.csv']
Mounted at /content/drive
copied to /content/drive/MyDrive/gas-repr-guide/results


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>